# 03. RFM Analysis

Calculate RFM scores and segment customers based on purchasing behavior.

**Output:** `data/processed/rfm_scores.csv`

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.rfm_calculator import run_rfm_analysis, calculate_rfm, assign_rfm_scores, assign_segments

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## Load Transaction Data

In [ ]:
transactions = pd.read_csv('../data/raw/ecommerce_transactions.csv')
print(f"Loaded {len(transactions):,} transactions from {transactions['customer_id'].nunique():,} customers")

## Run RFM Analysis

In [ ]:
rfm_df, segment_summary = run_rfm_analysis(
    transactions,
    save_path='../data/processed/rfm_scores.csv'
)

In [ ]:
rfm_df.head(10)

## RFM Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Recency distribution
axes[0].hist(rfm_df['recency'], bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Days Since Last Purchase')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Recency Distribution', fontweight='bold')
axes[0].axvline(rfm_df['recency'].median(), color='darkred', linestyle='--', label=f"Median: {rfm_df['recency'].median():.0f} days")
axes[0].legend()

# Frequency distribution
axes[1].hist(rfm_df['frequency'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Number of Purchases')
axes[1].set_ylabel('Number of Customers')
axes[1].set_title('Frequency Distribution', fontweight='bold')
axes[1].axvline(rfm_df['frequency'].median(), color='darkblue', linestyle='--', label=f"Median: {rfm_df['frequency'].median():.0f}")
axes[1].legend()

# Monetary distribution
axes[2].hist(rfm_df['monetary'], bins=50, color='green', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Total Spend ($)')
axes[2].set_ylabel('Number of Customers')
axes[2].set_title('Monetary Distribution', fontweight='bold')
axes[2].axvline(rfm_df['monetary'].median(), color='darkgreen', linestyle='--', label=f"Median: ${rfm_df['monetary'].median():,.0f}")
axes[2].legend()

plt.tight_layout()
plt.savefig('../reports/figures/rfm_distributions.png', dpi=150)
plt.show()

## RFM Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color in zip(axes, ['R_score', 'F_score', 'M_score'], ['coral', 'steelblue', 'green']):
    score_counts = rfm_df[col].value_counts().sort_index()
    bars = ax.bar(score_counts.index, score_counts.values, color=color, edgecolor='white')
    ax.set_xlabel('Score')
    ax.set_ylabel('Number of Customers')
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xticks([1, 2, 3, 4, 5])
    
    for bar, val in zip(bars, score_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, f'{val}', ha='center')

plt.tight_layout()
plt.savefig('../reports/figures/rfm_score_distributions.png', dpi=150)
plt.show()

## RFM Heatmap (R vs F)

In [ ]:
# Create pivot table for heatmap
rf_pivot = rfm_df.groupby(['R_score', 'F_score']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(rf_pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_xlabel('Frequency Score', fontsize=12)
ax.set_ylabel('Recency Score', fontsize=12)
ax.set_title('Customer Count by R-F Scores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/rf_heatmap.png', dpi=150)
plt.show()

## Customer Segments

In [ ]:
segment_summary

In [ ]:
# Segment distribution visualization
segment_counts = rfm_df['segment'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Customer count by segment
colors = sns.color_palette('husl', len(segment_counts))
bars = axes[0].barh(segment_counts.index, segment_counts.values, color=colors)
axes[0].set_xlabel('Number of Customers')
axes[0].set_title('Customers by Segment', fontweight='bold')
for bar, val in zip(bars, segment_counts.values):
    axes[0].text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center')

# Revenue by segment
segment_revenue = rfm_df.groupby('segment')['monetary'].sum().sort_values(ascending=False)
axes[1].pie(segment_revenue, labels=segment_revenue.index, autopct='%1.1f%%', colors=colors)
axes[1].set_title('Revenue Share by Segment', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/segment_distribution.png', dpi=150)
plt.show()

## Segment Characteristics Radar Chart

In [ ]:
from math import pi

# Prepare data for radar
segment_means = rfm_df.groupby('segment')[['R_score', 'F_score', 'M_score']].mean()
categories = ['Recency', 'Frequency', 'Monetary']
N = len(categories)

# Select top segments for clarity
top_segments = ['Champions', 'Loyal Customers', 'At Risk', 'Hibernating', 'New Customers']
segment_means = segment_means.loc[segment_means.index.isin(top_segments)]

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

colors = sns.color_palette('husl', len(segment_means))
for i, (segment, row) in enumerate(segment_means.iterrows()):
    values = row.values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=segment, color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 5)
ax.set_title('Segment Profiles (RFM Scores)', fontsize=14, fontweight='bold', y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('../reports/figures/segment_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## Actionable Insights

In [ ]:
print("=" * 60)
print("ACTIONABLE INSIGHTS BY SEGMENT")
print("=" * 60)

recommendations = {
    'Champions': '🏆 Reward with exclusive perks, early access, VIP programs',
    'Loyal Customers': '💎 Cross-sell premium products, offer membership rewards',
    'Potential Loyalists': '📈 Send upsell campaigns, recommend popular products',
    'New Customers': '👋 Welcome emails, onboarding sequence, first-purchase incentives',
    'At Risk': '⚠️ Re-engagement campaigns, personalized "We miss you" offers',
    "Can't Lose Them": '🚨 Urgent win-back campaigns, call from account manager',
    'Needs Attention': '👀 Monitor closely, send targeted offers',
    'About to Sleep': '😴 Reactivation discounts, feedback surveys',
    'Hibernating': '💤 Strong win-back offers, final attempt campaigns',
    'Regular': '📊 Maintain engagement, occasional promotions'
}

for segment in rfm_df['segment'].unique():
    count = len(rfm_df[rfm_df['segment'] == segment])
    revenue = rfm_df[rfm_df['segment'] == segment]['monetary'].sum()
    rec = recommendations.get(segment, '📋 Standard marketing')
    print(f"\n{segment} ({count:,} customers | ${revenue:,.0f} revenue):")
    print(f"   {rec}")

In [ ]:
print("\n✅ RFM Analysis Complete!")
print(f"📁 Saved to: data/processed/rfm_scores.csv")